# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dwaynemongaya/flyrank-ml-internship_dwaynemongaya/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two Paper Findings + My Methodology Questions

### Finding 1: Content Lifecycle — Growing vs. Declining

The research paper found that growing content was younger on average than declining content. Growing pages had an average age of 185 days, while declining pages averaged 228 days. Their average word counts were almost identical, suggesting that content age showed a clearer difference than content length.

**My methodology question:**

I would like to understand how the growing and declining labels were defined. Were they based on a specific percentage change in impressions over a particular time window? The paper explains that trend direction uses the last 30 days compared with the previous 30 days, with a decline greater than 10% classified as down. I would also ask whether the analysis accounted for seasonality and differences in client history, since these factors could influence the observed results.

### Finding 2: The Freshness Multiplier

The paper reported that pages updated 31–90 days earlier had a 5.43:1 growth-to-decline ratio. It also reported a separate comparison among pages older than 365 days, where recently refreshed pages showed higher health scores and impressions than pages last updated 181–360 days earlier.

**My methodology question:**

I would like to understand whether the recently refreshed pages and the older, less recently updated pages had similar starting performance before the comparison. Could other factors, such as existing search visibility, content quality, or client differences, explain some of the observed gap? I also noticed that the 361+ day freshness group contained only 802 pages, including just 21 declining pages. How did the researchers account for this small sample when interpreting the result?


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# Load the same starter dataset used in Week 5
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Use the same features and target from Week 5
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]

# Same baseline rule used in Week 5
baseline = (
    (X["impressions_90d"] >= 500).astype(int) * 50 +
    (X["avg_position"] <= 10).astype(int) * 50
)

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    order = np.argsort(-scores)
    return labels[order[:k]].mean()

def evaluate_split(X_train, X_test, y_train, y_test, split_name):
    model = DecisionTreeClassifier(
        max_depth=4,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)
    model_scores = model.predict_proba(X_test)[:, 1]

    baseline_scores = baseline.loc[X_test.index]

    for k in [20, 50, 100]:
        print(
            split_name,
            f"Precision@{k}",
            "Baseline:",
            round(precision_at_k(
                baseline_scores, y_test, k
            ), 3),
            "Decision Tree:",
            round(precision_at_k(
                model_scores, y_test, k
            ), 3)
        )

# BEFORE: Random-row split
X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

evaluate_split(
    X_train_random, X_test_random,
    y_train_random, y_test_random,
    "Random-row split"
)

# AFTER: Grouped-by-client split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

print(
    "Client overlap:",
    len(
        set(groups.iloc[train_idx]) &
        set(groups.iloc[test_idx])
    )
)

evaluate_split(
    X_train_group, X_test_group,
    y_train_group, y_test_group,
    "Grouped-client split"
)


FileNotFoundError: [Errno 2] No such file or directory: '/content/content_refresh_anonymized.csv'

### Interpretation

I compared a random-row split with a grouped-by-client split using the same Decision Tree, features, baseline rule, and Precision@K metrics.

The random-row split allows observations from the same client to appear in both training and testing. This can produce an overly optimistic evaluation because the model may learn patterns associated with clients it has already encountered.

The grouped-by-client split keeps each client entirely within either the training or test set. This provides a more realistic assessment of whether the model's learned patterns generalize to unseen clients.

I will use the grouped-by-client results as my more cautious evaluation. Any difference between the two results shows why validation design matters when interpreting model performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage Audit

I reviewed the six features used by my Week-5 Decision Tree.

| Feature | Role | Leakage concern |
|---|---|---|
| content_age_days | Feature | Generally safe as content metadata, but its measurement date should be confirmed. |
| days_since_last_update | Feature | Generally safe if measured at the decision time. |
| impressions_90d | Feature | Potential concern because its time window may overlap with the trend label. |
| avg_position | Feature | Potential concern because it may describe the same period used to calculate the label. |
| ctr | Feature | Potential concern because it may be derived from measurements in the same period as the label. |
| word_count | Feature | Generally safe if measured before the decision time. |
| trend_direction | Target source | Used to construct the target, so it must not be included as a feature. |
| trend_pct | Derived outcome | Excluded because it directly represents the trend used to construct the target. |

The dataset does not include FlyRank's own product decision flags, so the model is not directly learning from those decisions.

However, I identified a more important limitation: my target is derived from `trend_direction`, which is calculated from the available trend measurements. Some of my input features use the same 90-day performance window.

This creates a possible target-window overlap. Although I excluded `trend_direction` and `trend_pct` from the input features, I cannot claim that all temporal leakage risks have been eliminated.

A stronger future experiment should use earlier observations as features and a separate future period as the target.

**Conclusion:** The model excludes the direct target and its percentage representation, but its current same-window design still has a temporal leakage risk.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim Rewrite

### Original claim

The Decision Tree can predict which pages will decline and identify the best pages to refresh.

### Revised claim

In this experiment, the Decision Tree showed measurable ability to identify pages associated with the current declining-trend proxy using the available content and search signals.

Its performance was evaluated using Precision@K and client-grouped validation. However, the target represents an observed trend within the available data rather than a separately measured future outcome.

Therefore, the model should be treated as a decision-support tool for prioritizing pages for human review, not as proof that a page will decline or that refreshing it will improve performance.

### What I can claim

- I observed measurable patterns between the selected features and the declining-trend proxy.
- I evaluated the model using a grouped-by-client split.
- I compared the Decision Tree against a fixed baseline.
- The model can support the prioritization of pages for human review.

### What I cannot claim

- That the model accurately predicts future Google rankings.
- That content refreshes will automatically improve search performance.
- That the selected features cause a page to decline.
- That the results generalize to every website or client.

My results should be interpreted as an initial experiment using an anonymized starter dataset.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.